# Paper 3 — M1: Static FR1 Scene and Measurement-Report Generator

**Milestone M1** (see `paper3_sionna_scene_spec.md` v0.1): static FR1 Munich scene, 8 distributed arrays,
synthetic measurement-report generator, HDF5 export in `wh-rdcc` tensor conventions, and the
sim-to-real **sanity gate** on delay-spread / K-factor / cluster-size statistics.

Pipeline: setup → Drive canary → config → scene build → TX placement check → batched ray tracing
(checkpointed) → derived stats & reports → HDF5 export → M1 gate → zip backup.

Runtime: Colab **GPU (T4)**. Each batch checkpoints to Drive; interrupted runs resume automatically.

In [ ]:
# %% [1] Environment setup — safe to re-run; auto-restarts ONCE after installing
import importlib.util, subprocess, sys

if importlib.util.find_spec('sionna') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'sionna==2.0.1', 'h5py'], check=True)
    # pip replaced numpy on disk while the old build was loaded:
    # a clean restart is the only reliable way to avoid mixed-version imports.
    print('Install done — the runtime will now restart (EXPECTED).')
    print('When it comes back, run this cell again: it will skip the install.')
    import os, time; time.sleep(1); os.kill(os.getpid(), 9)

!nvidia-smi -L
import sionna, sionna.rt
print('Sionna:', sionna.__version__, '| Sionna RT:', sionna.rt.__version__)

In [ ]:
# %% [2] Drive mount + canary (mandatory before any save)
import os, uuid
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/paper3'
M1_DIR = os.path.join(BASE, 'm1')
CKPT_DIR = os.path.join(M1_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

def drive_canary(path=M1_DIR):
    """Write-read-delete canary: catches phantom mounts before we trust Drive."""
    token = str(uuid.uuid4())
    canary = os.path.join(path, '.canary')
    with open(canary, 'w') as f:
        f.write(token)
    with open(canary) as f:
        assert f.read() == token, 'DRIVE CANARY FAILED: phantom mount, do NOT save.'
    os.remove(canary)
    print('Drive canary OK:', path)

drive_canary()

In [ ]:
# %% [3] Configuration (single source of truth — mirrors scene spec v0.1)
from dataclasses import dataclass, field
import numpy as np

@dataclass
class M1Config:
    # --- band (FR1, DICHASUS-matched bandwidth) ---
    carrier_hz: float = 3.5e9
    bandwidth_hz: float = 50e6
    num_subcarriers: int = 64                      # decimated grid, as in WARDEN
    # --- geometry ---
    area_center: tuple = (0.0, 50.0)               # Munich scene coords; adjust after preview
    area_half: float = 100.0                       # working area: 200 m x 200 m
    ue_height: float = 1.5
    tx_height: float = 7.0                         # lamppost height
    # Two-ring deployment (12 arrays): outer ring covers the perimeter, inner ring the center.
    # Density chosen so that typical serving clusters have 2-4 arrays (spec section 2).
    n_outer: int = 8
    n_inner: int = 4                               # set to 0 for the A8 geometry (pre-registered A8 vs A12 comparison)
    tx_ring_radius: float = 80.0
    inner_ring_radius: float = 35.0
    tx_ring_jitter: float = 12.0
    # --- ray tracing ---
    max_depth: int = 5
    # --- sampling ---
    n_points: int = 5000                           # OUTDOOR UE positions (rejection-sampled, see cell 6)
    batch_size: int = 48                           # receivers per solve (T4-sized)
    # --- reports ---
    gamma_db: float = -10.0                        # array in cluster if it delivers >= 10% of strongest array's energy
    top_k: tuple = (2, 3, 4)                       # ablation variants
    rsrp_quant_db: float = 1.0
    toa_quant_s: float = 1.0 / 50e6                # 20 ns = 1/BW
    max_neighbors: int = 8
    # --- reproducibility ---
    seed: int = 2026

CFG = M1Config()
RNG = np.random.default_rng(CFG.seed)
SC_SPACING = CFG.bandwidth_hz / CFG.num_subcarriers
print(f'Subcarrier spacing: {SC_SPACING/1e3:.2f} kHz | delay resolution: {CFG.toa_quant_s*1e9:.0f} ns')

In [ ]:
# %% [4] Scene build: Munich, 8 distributed arrays (VAA), FR1 antenna configs
import mitsuba as mi
from sionna.rt import (load_scene, PlanarArray, Transmitter, Receiver,
                       PathSolver, subcarrier_frequencies)

scene = load_scene(sionna.rt.scene.munich, merge_shapes=True)
scene.frequency = CFG.carrier_hz

# Spec section 3: array = 4x2 planar single-pol; UE = 2 omni elements
scene.tx_array = PlanarArray(num_rows=2, num_cols=4,
                             vertical_spacing=0.5, horizontal_spacing=0.5,
                             pattern='tr38901', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=2,
                             vertical_spacing=0.5, horizontal_spacing=0.5,
                             pattern='iso', polarization='V')

def ring(n, radius, offset, cfg, rng):
    ang = np.linspace(0, 2*np.pi, n + 1)[:-1] + offset + rng.uniform(-0.15, 0.15, n)
    r = radius + rng.uniform(-cfg.tx_ring_jitter, cfg.tx_ring_jitter, n)
    return np.stack([cfg.area_center[0] + r*np.cos(ang),
                     cfg.area_center[1] + r*np.sin(ang),
                     np.full(n, cfg.tx_height)], axis=1)

def tx_positions(cfg, rng):
    outer = ring(cfg.n_outer, cfg.tx_ring_radius, 0.0, cfg, rng)
    inner = ring(cfg.n_inner, cfg.inner_ring_radius, np.pi/cfg.n_inner, cfg, rng)  # staggered vs outer
    return np.concatenate([outer, inner], axis=0)

TX_POS = tx_positions(CFG, np.random.default_rng(CFG.seed))  # dedicated stream: stable placement
for a, p in enumerate(TX_POS):
    tx = Transmitter(name=f'tx_{a}', position=[float(p[0]), float(p[1]), float(p[2])])
    scene.add(tx)
    tx.look_at([CFG.area_center[0], CFG.area_center[1], CFG.ue_height])  # boresight to area center

A = TX_POS.shape[0]
FREQS = subcarrier_frequencies(CFG.num_subcarriers, SC_SPACING)
SOLVER = PathSolver()
print('Arrays placed:\n', np.round(TX_POS, 1))

In [ ]:
# %% [5] TX placement sanity check
# Munich geometry is irregular: a TX inside a building yields (near-)zero paths.
# Probe each TX against a small ring of test receivers and flag suspicious ones,
# then render the scene for visual confirmation. Edit area_center / ring params if flagged.
probe_names = []
for i in range(12):
    ang = 2*np.pi*i/12
    p = [CFG.area_center[0] + 40*np.cos(ang), CFG.area_center[1] + 40*np.sin(ang), CFG.ue_height]
    name = f'probe_{i}'
    scene.add(Receiver(name=name, position=[float(v) for v in p]))
    probe_names.append(name)

paths = SOLVER(scene, max_depth=CFG.max_depth)
a_pr, tau_pr = paths.cir(normalize_delays=False, out_type='numpy')
tau_link = tau_pr if tau_pr.ndim == 3 else tau_pr[:, 0, :, 0, :]
hits = (tau_link >= 0).sum(axis=(0, 2)) if tau_link.shape[1] == A else (tau_link >= 0).sum(axis=(1, 2))
for name in probe_names:
    scene.remove(name)

for a_idx in range(A):
    status = 'OK' if hits[a_idx] > 0 else '*** SUSPICIOUS: no paths — likely inside a building ***'
    print(f'tx_{a_idx}: {int(hits[a_idx]):5d} probe paths  {status}')

from sionna.rt import Camera
cam = Camera(position=[CFG.area_center[0], CFG.area_center[1]-250, 200],
             look_at=[CFG.area_center[0], CFG.area_center[1], 0])
scene.render(camera=cam)  # visual check of TX placement

In [ ]:
# %% [5b] Relocate weak TXs (relative threshold) and re-probe
def probe_hits():
    names = []
    for i in range(12):
        ang = 2*np.pi*i/12
        p = [CFG.area_center[0] + 40*np.cos(ang), CFG.area_center[1] + 40*np.sin(ang), CFG.ue_height]
        n = f'probe_{i}'
        scene.add(Receiver(name=n, position=[float(v) for v in p]))
        names.append(n)
    paths = SOLVER(scene, max_depth=CFG.max_depth)
    _, tau = paths.cir(normalize_delays=False, out_type='numpy')
    tau = np.asarray(tau)
    t = tau if tau.ndim == 3 else tau[:, 0, :, 0, :]
    for n in names:
        scene.remove(n)
    return (t >= 0).sum(axis=(0, 2)) if t.shape[1] == A else (t >= 0).sum(axis=(1, 2))

REL_THR = 0.25
center = [CFG.area_center[0], CFG.area_center[1], CFG.ue_height]
hits = probe_hits()
thr = REL_THR * np.median(hits)
weak = np.where(hits < thr)[0]
print(f'Median hits: {np.median(hits):.0f} | threshold: {thr:.0f} | weak TXs: {weak.tolist()}')

for a_idx in weak:
    tx = scene.get(f'tx_{a_idx}')
    x0, y0 = TX_POS[a_idx, 0] - CFG.area_center[0], TX_POS[a_idx, 1] - CFG.area_center[1]
    ang0, r0 = np.arctan2(y0, x0), np.hypot(x0, y0)
    best_h, best_p = hits[a_idx], TX_POS[a_idx].copy()
    for dr in (-30, -20, -10, 10, 20):
        for dang in (-0.15, 0.0, 0.15):
            p = np.array([CFG.area_center[0] + (r0+dr)*np.cos(ang0+dang),
                          CFG.area_center[1] + (r0+dr)*np.sin(ang0+dang), CFG.tx_height])
            tx.position = mi.Point3f(*map(float, p)); tx.look_at(center)
            h = probe_hits()[a_idx]
            if h > best_h:
                best_h, best_p = h, p
    tx.position = mi.Point3f(*map(float, best_p)); tx.look_at(center)
    TX_POS[a_idx] = best_p
    flag = '' if best_h >= thr else '  *** still weak: move manually ***'
    print(f'tx_{a_idx}: {hits[a_idx]} -> {best_h} probe paths at {np.round(best_p, 1)}{flag}')

final = probe_hits()
print('Final probe hits:', final.tolist())
drive_canary(M1_DIR)
np.save(os.path.join(M1_DIR, 'tx_positions.npy'), TX_POS)
scene.render(camera=cam)

In [ ]:
# %% [6] UE sampling — OUTDOOR ONLY (rejection sampling), seeded and geometry-independent
# An upward ray that hits geometry means the point is under a roof, i.e. inside a building.
# M1 v1 sampled uniformly and put 45.5% of UEs indoors: invalid for a pedestrian-street model.
def is_indoor(pos):
    o = mi.Point3f(mi.Float(pos[:, 0]), mi.Float(pos[:, 1]), mi.Float(pos[:, 2]))
    si = scene.mi_scene.ray_intersect(mi.Ray3f(o, mi.Vector3f(0, 0, 1)))
    return np.array(si.is_valid(), dtype=bool)

UE_RNG = np.random.default_rng(CFG.seed + 1)   # dedicated stream: the SAME UE set for A8 and A12 (paired comparison)
chunks, n_have, n_drawn = [], 0, 0
while n_have < CFG.n_points:
    cand = UE_RNG.uniform(-CFG.area_half, CFG.area_half, size=(2*CFG.n_points, 2)) + np.array(CFG.area_center)
    cand = np.concatenate([cand, np.full((len(cand), 1), CFG.ue_height)], axis=1).astype(np.float32)
    keep = cand[~is_indoor(cand)]
    chunks.append(keep); n_have += len(keep); n_drawn += len(cand)
UE_POS = np.concatenate(chunks)[:CFG.n_points]
assert not is_indoor(UE_POS).any(), 'indoor UE leaked into the outdoor set'
print(f'Outdoor UEs: {len(UE_POS)} | acceptance rate {n_have/n_drawn*100:.1f}%')

RUN_TAG = f'A{A}_N{CFG.n_points}_outdoor_seed{CFG.seed}'   # geometry-specific: never mix runs
np.save(os.path.join(M1_DIR, f'ue_positions_{RUN_TAG}.npy'), UE_POS)
print('Run tag:', RUN_TAG)

In [ ]:
# %% [7] Batched ray tracing with per-batch Drive checkpoints (resume-safe)
import glob, time

def solve_batch(positions):
    names = []
    for i, p in enumerate(positions):
        name = f'rx_{i}'
        scene.add(Receiver(name=name, position=[float(p[0]), float(p[1]), float(p[2])]))
        names.append(name)
    paths = SOLVER(scene, max_depth=CFG.max_depth)
    # CFR on the 64-subcarrier grid: [B, rx_ant, A, tx_ant, T, S] -> squeeze T
    h = paths.cfr(frequencies=FREQS, normalize_delays=False, out_type='numpy')
    h = np.asarray(h).squeeze(-2).astype(np.complex64)
    a_cir, tau = paths.cir(normalize_delays=False, out_type='numpy')
    a_cir, tau = np.asarray(a_cir), np.asarray(tau)
    for n in names:
        scene.remove(n)
    # Per-link path power and absolute delays
    p_path = (np.abs(a_cir[..., 0])**2).sum(axis=(1, 3))          # [B, A, P]
    tau_link = tau if tau.ndim == 3 else tau[:, 0, :, 0, :]        # [B, A, P]
    return h, p_path.astype(np.float32), tau_link.astype(np.float32)

RUN_CKPT = os.path.join(CKPT_DIR, RUN_TAG); os.makedirs(RUN_CKPT, exist_ok=True)
n_batches = int(np.ceil(CFG.n_points / CFG.batch_size))
done = {int(f.split('_')[-1].split('.')[0]) for f in glob.glob(os.path.join(RUN_CKPT, 'batch_*.npz'))}
print(f'[{RUN_TAG}] {n_batches} batches | {len(done)} already checkpointed (resume)')

for b in range(n_batches):
    if b in done:
        continue
    t0 = time.time()
    sl = slice(b*CFG.batch_size, min((b+1)*CFG.batch_size, CFG.n_points))
    h, p_path, tau_link = solve_batch(UE_POS[sl])
    drive_canary(RUN_CKPT)  # never trust the mount blindly before a save
    np.savez_compressed(os.path.join(RUN_CKPT, f'batch_{b:04d}.npz'),
                        h=h, p_path=p_path, tau=tau_link, idx=np.arange(sl.start, sl.stop))
    print(f'batch {b+1}/{n_batches}  {time.time()-t0:5.1f} s  csi {h.shape}')

In [ ]:
# %% [8] Assemble + derived quantities: energies, clusters, reports
files = sorted(glob.glob(os.path.join(RUN_CKPT, 'batch_*.npz')))
H, PP, TAU, IDX = [], [], [], []
for f in files:
    z = np.load(f)
    H.append(z['h']); PP.append(z['p_path']); TAU.append(z['tau']); IDX.append(z['idx'])
# Max path count varies per solve: pad the path axis (power -> 0, delay -> -1 = "no path")
P_MAX = max(p.shape[-1] for p in PP)
def pad_paths(x, fill):
    return np.pad(x, ((0, 0), (0, 0), (0, P_MAX - x.shape[-1])), constant_values=fill)
H = np.concatenate(H)
PP = np.concatenate([pad_paths(p, 0.0) for p in PP])
TAU = np.concatenate([pad_paths(t, -1.0) for t in TAU])
order = np.argsort(np.concatenate(IDX)); H, PP, TAU = H[order], PP[order], TAU[order]
print('CSI tensor:', H.shape, '| expected [N, rx_ant, A, tx_ant, S] | path axis padded to', P_MAX)

valid = TAU >= 0                                             # Sionna marks absent paths with tau = -1
# Wideband energy per (UE, array), from the CFR
E_lin = (np.abs(H)**2).sum(axis=(1, 3, 4)) + 1e-30           # [N, A]
E_dB = 10*np.log10(E_lin)
covered = E_dB.max(axis=1) > (E_dB.max() - 80)               # drop UEs with no meaningful coverage
print(f'Coverage: {covered.mean()*100:.1f}% of points ({(~covered).sum()} dropped)')

# RMS delay spread & K-factor per link (path domain)
w = np.where(valid, PP, 0.0)
w_sum = w.sum(-1) + 1e-30
tau_mean = (w*np.where(valid, TAU, 0)).sum(-1)/w_sum
ds_rms = np.sqrt((w*(np.where(valid, TAU, 0)-tau_mean[..., None])**2).sum(-1)/w_sum)   # [N, A]
p_max = w.max(-1)
has_paths = valid.any(-1)                                     # links with at least one path
with np.errstate(divide='ignore'):
    k_factor_dB = np.where(has_paths,
                           10*np.log10(np.maximum(p_max, 1e-30)/np.maximum(w_sum - p_max, 1e-30)),
                           np.nan)                            # NaN for no-path links (never in a cluster)

# Cluster ground truth: gamma-threshold (primary) + top-K variants
members_gamma = (E_dB >= E_dB.max(axis=1, keepdims=True) + CFG.gamma_db)               # [N, A] bool
members_topk = {k: np.zeros_like(members_gamma) for k in CFG.top_k}
rank = np.argsort(-E_dB, axis=1)
for k in CFG.top_k:
    np.put_along_axis(members_topk[k], rank[:, :k], True, axis=1)

# Measurement reports (3GPP-flavored quantization)
E_q = np.round(E_dB/CFG.rsrp_quant_db)*CFG.rsrp_quant_db
toa = np.where(valid, TAU, np.inf).min(-1)                                             # first-path ToA [N, A]
strongest = E_dB.argmax(axis=1)
with np.errstate(invalid='ignore'):
    dtau = toa - np.take_along_axis(toa, strongest[:, None], axis=1)                   # relative ToA (TDoA-like)
dtau = np.where(np.isfinite(dtau), dtau, np.nan)                                       # NaN = neighbor not heard
dtau_q = np.round(dtau/CFG.toa_quant_s)*CFG.toa_quant_s
neighbor_mask = np.zeros_like(members_gamma)
np.put_along_axis(neighbor_mask, rank[:, :CFG.max_neighbors], True, axis=1)
print('Cluster size (gamma) distribution:',
      dict(zip(*np.unique(members_gamma[covered].sum(1), return_counts=True))))
print('Cluster membership frequency per array:',
      np.round(members_gamma[covered].mean(0)*100, 1).tolist(), '%')

In [ ]:
# %% [8b] Diagnostic: cluster-size distribution vs gamma (no re-tracing)
Ec = E_dB[covered]
for g in (-6, -8, -10, -12, -15):
    s = (Ec >= Ec.max(1, keepdims=True) + g).sum(1)
    print(f'gamma {g:4d} dB | singletons {np.mean(s==1)*100:5.1f}% | size 2-4 {np.isin(s,[2,3,4]).mean()*100:5.1f}% '
          f'| mean {s.mean():.2f} | max {s.max()}')

In [ ]:
# %% [9] HDF5 export — wh-rdcc-compatible layout (spec section 6)
import h5py, json, datetime

drive_canary(M1_DIR)
out = os.path.join(M1_DIR, f'paper3_static_fr1_{RUN_TAG}.h5')
with h5py.File(out, 'w') as f:
    f.create_dataset('csi', data=H[covered], compression='gzip')                # [N, rx_ant, A, tx_ant, S]
    f.create_dataset('positions', data=UE_POS[covered], compression='gzip')    # ground truth, never for training
    f.create_dataset('tx_positions', data=TX_POS.astype(np.float32))          # array geometry (needed for E3)
    rep = f.create_group('reports')
    rep.create_dataset('E_dB_q', data=E_q[covered].astype(np.float32))
    rep.create_dataset('dtau_q', data=dtau_q[covered].astype(np.float32))
    rep.create_dataset('sigma_tau', data=ds_rms[covered].astype(np.float32))
    rep.create_dataset('neighbor_mask', data=neighbor_mask[covered])
    cg = f.create_group('cluster_gt')
    cg.create_dataset('gamma', data=members_gamma[covered])
    for k, m in members_topk.items():
        cg.create_dataset(f'top{k}', data=m[covered])
    f.attrs.update(dict(band='FR1', regime='static', scene='munich',
                        carrier_hz=CFG.carrier_hz, bandwidth_hz=CFG.bandwidth_hz,
                        gamma_db=CFG.gamma_db, seed=CFG.seed,
                        sionna=sionna.__version__,
                        created=datetime.datetime.now().isoformat()))
print('Saved:', out, f'({os.path.getsize(out)/1e6:.1f} MB)')

In [ ]:
# %% [10] M1 sanity gate — sim-to-real statistics
import matplotlib.pyplot as plt

ds_ns = ds_rms[covered][members_gamma[covered]] * 1e9        # serving-cluster links only
kf = k_factor_dB[covered][members_gamma[covered]]
csize = members_gamma[covered].sum(1)

# Gate bands (EDIT after computing the DICHASUS reference with the wh-rdcc pipeline).
# NOTE: cf0x is indoor, Munich is outdoor urban -> a level shift in delay spread is expected;
# the gate is order-of-magnitude + distribution shape, not exact matching.
# Revised criteria (spec v0.2): the v0.1 target "60% of clusters of size 2-4" proved unattainable in mixed
# LoS/NLoS urban geometry for any gamma or density. The criteria below test what the paper needs:
# multi-array geometry for most UEs, and non-degenerate (non-saturated) clusters.
GATE = dict(ds_median_ns=(30, 300), singleton_max=0.20, median_size_frac_max=0.5, coverage_min=0.95)
ref_path = os.path.join(BASE, 'dichasus_ref_stats.json')     # optional, produced offline from cf0x
ref = json.load(open(ref_path)) if os.path.exists(ref_path) else None

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(ds_ns, bins=60, cumulative=True, density=True, histtype='step', label='sim FR1 static')
if ref: ax[0].hist(ref['ds_ns'], bins=60, cumulative=True, density=True, histtype='step', label='DICHASUS cf0x')
ax[0].set_xlabel('RMS delay spread [ns]'); ax[0].set_ylabel('CDF'); ax[0].legend()
ax[1].hist(kf, bins=60, density=True); ax[1].set_xlabel('K-factor [dB]')
ax[2].bar(*np.unique(csize, return_counts=True)); ax[2].set_xlabel('cluster size (gamma)')
plt.tight_layout(); plt.savefig(os.path.join(M1_DIR, 'm1_gate.png'), dpi=150); plt.show()

sing = np.mean(csize == 1)
se = np.sqrt(sing*(1-sing)/len(csize))
checks = {
    'DS median in band': GATE['ds_median_ns'][0] <= np.median(ds_ns) <= GATE['ds_median_ns'][1],
    'singletons <= 20%': sing <= GATE['singleton_max'],
    'clusters not saturated (median <= A/2)': np.median(csize) <= GATE['median_size_frac_max']*A,
    'coverage': covered.mean() >= GATE['coverage_min'],
}
print(f'[{RUN_TAG}] DS median: {np.median(ds_ns):.0f} ns | P90: {np.percentile(ds_ns, 90):.0f} ns | '
      f'K median: {np.nanmedian(kf):.1f} dB')
print(f'Singletons: {sing*100:.1f}% (SE {se*100:.1f} pp) | size 2-4: {np.isin(csize,[2,3,4]).mean()*100:.1f}% | '
      f'median size: {np.median(csize):.0f} | P90: {np.percentile(csize,90):.0f}')
for k, ok in checks.items():
    print(f'[{"PASS" if ok else "FAIL"}] {k}')
print('\nM1 GATE:', 'PASSED' if all(checks.values()) else 'NOT PASSED — revisit placement/materials before M2')

In [ ]:
# %% [11] Zip backup to PC (end-of-session discipline)
import shutil
zip_path = shutil.make_archive('/content/paper3_m1_backup', 'zip', M1_DIR)
from google.colab import files as colab_files
colab_files.download(zip_path)